<a href="https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The following questions are constructive. The paper already discloses that it is observational and that its headline findings prioritize direct aggregate comparisons. My questions identify the additional validation needed before interpreting an observed association as causal or broadly generalizable.

### Finding 1 — The Anatomy of Growing Content

**What the paper reports:** Pages labelled as growing averaged approximately 3,180 words and 184 days of age, while declining pages averaged approximately 2,311 words and 230 days. The reported cohorts contained approximately 74.2K growing and 45.3K declining pages.

**Where the label comes from:** The paper defines `Trend Direction` using percentage change in impressions during the most recent 30 days compared with the previous 30 days. `Up` represents growth greater than +10%, `down` represents decline below -10%, and `stable` represents values within ±10%. Therefore, this label measures short-window impression momentum. It does not directly measure content quality or the effect of changing a page.

**My methodology question:** How sensitive are the up/down labels to small previous-30-day impression denominators, client scale, content type, and seasonality? Do the word-count and age differences remain when pages are compared within the same client and within similar baseline-impression, content-type, and age groups?

A grouped analysis by client, minimum-impression requirements, uncertainty intervals, and replication in a later time window would strengthen the directional finding.

### Finding 4 — The Freshness Multiplier

**What the paper reports:** Among content aged 365+ days, pages refreshed within 30 days showed a 3.2× health-score difference, from 10.7 to 34.5, and a 57× impression difference, from 71 to 4,039. The paper also warns that the `361+` freshness bucket is unstable because it contains only one declining page.

**Where the exposure and outcomes come from:** Freshness is measured using days since the last update. Health is a FlyRank composite containing impressions, position, CTR, and scroll depth. Impressions come from the rolling performance snapshot. This is an observed comparison; pages were not randomly assigned to receive a refresh.

**My methodology question:** Was every refresh completed before the performance-outcome window, and were refreshed pages compared with similarly visible mature pages that were not refreshed? Editors may preferentially refresh high-value pages, creating selection and survivor bias.

A stronger design would match pages by client, baseline visibility, age, and word count, then compare forward 30-day and 60-day changes for refreshed versus untreated pages. A difference-in-differences design, sample sizes, and uncertainty intervals would support a claim about measured post-refresh change. The current evidence supports an observed portfolio association.

In [5]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os

import duckdb
import numpy as np
import pandas as pd
import sklearn

from IPython.display import Markdown, display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 200)

paper_audit = pd.DataFrame([
    {
        "paper_finding": "#1 — Anatomy of Growing Content",
        "reported_observation":
            "Growing: 3,180 words / 184 days; "
            "declining: 2,311 words / 230 days",
        "methodology_focus":
            "Trend-label denominator, within-client confounding, "
            "minimum exposure, and out-of-time replication",
        "supported_claim_level":
            "Observed directional association",
    },
    {
        "paper_finding": "#4 — Freshness Multiplier",
        "reported_observation":
            "365+ pages refreshed within 30 days: "
            "3.2x health and 57x impressions",
        "methodology_focus":
            "Refresh timing, matched controls, selection bias, "
            "and forward outcomes",
        "supported_claim_level":
            "Observed portfolio association",
    },
])

display(paper_audit)

assert len(paper_audit) == 2
assert paper_audit[
    "supported_claim_level"
].str.contains("Observed").all()

print(
    "PASS: Two paper findings were named and "
    "questioned constructively."
)

,paper_finding,reported_observation,methodology_focus,supported_claim_level
0,#1 — Anatomy of Growing Content,"Growing: 3,180 words / 184 days; declining: 2,311 words / 230 days","Trend-label denominator, within-client confounding, minimum exposure, and out-of-time replication",Observed directional association
1,#4 — Freshness Multiplier,365+ pages refreshed within 30 days: 3.2x health and 57x impressions,"Refresh timing, matched controls, selection bias, and forward outcomes",Observed portfolio association


PASS: Two paper findings were named and questioned constructively.


## 2. My model under an honest split — before/after

My Week-5 notebook already used a client-grouped holdout. To show the validation improvement clearly, I recreate a random row split only as the diagnostic **before** case. I then rerun the same Logistic Regression with the same five historical features using the client-grouped holdout as the honest **after** case.

The random row split allows pages belonging to the same clients to appear in both training and test data. Client-specific traffic scale and content patterns can therefore cross the validation boundary.

The grouped split places every page belonging to one client entirely in either training or test data. It asks the harder and more useful question: does the measured ranking transfer to clients the model has not seen?

The improvement is in the validation design, not necessarily in the metric. The test populations have different row counts and base rates, so every score is displayed beside its test base rate. A weaker grouped result is evidence about limited cross-client generalization; it is not a reason to prefer the easier random split.

In [6]:
# ---------------------------------------------------------
# Secure Hugging Face connection
# ---------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, (
    "HF_TOKEN is missing. Open Colab Secrets, add a "
    "Hugging Face READ token named HF_TOKEN, enable "
    "notebook access, and run all cells again."
)

safe_token = HF_TOKEN.replace("'", "''")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

# ---------------------------------------------------------
# Rebuild the exact Week-5 feature frame
# ---------------------------------------------------------

feature_sql = f"""
WITH aggregated AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_days,

        COUNT(DISTINCT report_date) FILTER (
            WHERE report_date >= DATE '2026-03-22'
        ) AS outcome_days,

        SUM(gsc_impressions) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_impressions,

        SUM(gsc_clicks) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_clicks,

        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '2026-03-22'
        ) AS outcome_impressions,

        (
            SUM(
                gsc_avg_position * gsc_impressions
            ) FILTER (
                WHERE report_date < DATE '2026-03-22'
            )
            /
            NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date < DATE '2026-03-22'
                ),
                0
            )
        ) AS hist_avg_position,

        STDDEV_SAMP(gsc_avg_position) FILTER (
            WHERE report_date < DATE '2026-03-22'
              AND gsc_impressions > 0
        ) AS hist_position_sd

    FROM {MARCH_DAILY}

    WHERE ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

framed AS (
    SELECT
        *,

        1.0 * hist_impressions
            / NULLIF(hist_days, 0)
            AS hist_imp_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_days, 0)
            AS hist_click_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_impressions, 0)
            AS hist_ctr,

        1.0 * outcome_impressions
            / NULLIF(outcome_days, 0)
            AS outcome_imp_per_day

    FROM aggregated

    WHERE hist_days >= 14
      AND outcome_days >= 7
      AND hist_impressions >= 50
)

SELECT
    client_hash_id,
    content_hash_id,
    hist_days,
    outcome_days,
    hist_imp_per_day,
    hist_click_per_day,
    hist_ctr,
    hist_avg_position,
    hist_position_sd,

    CAST(
        outcome_imp_per_day
        < 0.8 * hist_imp_per_day
        AS INTEGER
    ) AS is_declining_proxy

FROM framed
"""

raw_frame = con.sql(feature_sql).df()

FEATURES = [
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

LABEL = "is_declining_proxy"
GROUP = "client_hash_id"
CONTENT_ID = "content_hash_id"

model_frame = (
    raw_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=FEATURES
        + [LABEL, GROUP, CONTENT_ID]
    )
    .copy()
)

model_frame[LABEL] = (
    model_frame[LABEL].astype(int)
)

assert len(model_frame) == 2348
assert model_frame[GROUP].nunique() == 14
assert model_frame[LABEL].nunique() == 2

assert (
    model_frame.duplicated(
        [GROUP, CONTENT_ID]
    ).sum()
    == 0
)

assert np.isclose(
    model_frame[LABEL].mean(),
    0.2355,
    atol=0.0001,
)

frame_summary = pd.DataFrame([{
    "eligible_rows": len(model_frame),
    "pseudonymous_clients":
        model_frame[GROUP].nunique(),
    "overall_positive_base_rate":
        model_frame[LABEL].mean(),
    "feature_count": len(FEATURES),
    "duplicate_client_content_rows":
        model_frame.duplicated(
            [GROUP, CONTENT_ID]
        ).sum(),
    "queried_partition": "2026-03 only",
}])

display(frame_summary.round(4))

print("June/_sample queried: NO")


# ---------------------------------------------------------
# Model and ranking functions
# ---------------------------------------------------------

def make_logistic_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ])


def rank_for_review(
    frame,
    score_column="model_score",
):
    return (
        frame
        .sort_values(
            [
                score_column,
                "hist_imp_per_day",
                "hist_click_per_day",
                CONTENT_ID,
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


def precision_at_k(
    scored_frame,
    k,
):
    ranked = rank_for_review(
        scored_frame
    )

    effective_k = min(
        k,
        len(ranked),
    )

    return float(
        ranked
        .head(effective_k)[LABEL]
        .mean()
    )


# ---------------------------------------------------------
# BEFORE: random row split
# ---------------------------------------------------------

all_indices = np.arange(
    len(model_frame)
)

row_train_idx, row_test_idx = (
    train_test_split(
        all_indices,
        test_size=0.25,
        random_state=SEED,
        stratify=model_frame[LABEL],
    )
)


# ---------------------------------------------------------
# AFTER: client-grouped holdout
# ---------------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        model_frame[FEATURES],
        model_frame[LABEL],
        groups=model_frame[GROUP],
    )
)

random_name = (
    "Before — random row split "
    "(diagnostic only)"
)

grouped_name = (
    "After — client-grouped "
    "holdout (honest)"
)

split_indices = {
    random_name:
        (row_train_idx, row_test_idx),

    grouped_name:
        (group_train_idx, group_test_idx),
}


# ---------------------------------------------------------
# Fit exactly the same model on each split
# ---------------------------------------------------------

split_runs = {}
metric_rows = []

for split_name, (
    train_idx,
    test_idx,
) in split_indices.items():

    train_frame = (
        model_frame
        .iloc[train_idx]
        .reset_index(drop=True)
    )

    test_frame = (
        model_frame
        .iloc[test_idx]
        .reset_index(drop=True)
    )

    train_clients = set(
        train_frame[GROUP]
    )

    test_clients = set(
        test_frame[GROUP]
    )

    client_overlap = len(
        train_clients.intersection(
            test_clients
        )
    )

    assert train_frame[LABEL].nunique() == 2
    assert test_frame[LABEL].nunique() == 2

    model = make_logistic_model()

    model.fit(
        train_frame[FEATURES],
        train_frame[LABEL],
    )

    scored_test = test_frame.copy()

    scored_test["model_score"] = (
        model.predict_proba(
            test_frame[FEATURES]
        )[:, 1]
    )

    base_rate = float(
        scored_test[LABEL].mean()
    )

    p20 = precision_at_k(
        scored_test,
        20,
    )

    p50 = precision_at_k(
        scored_test,
        50,
    )

    metric_rows.append({
        "split_design": split_name,
        "train_rows": len(train_frame),
        "test_rows": len(test_frame),
        "train_clients":
            len(train_clients),
        "test_clients":
            len(test_clients),
        "client_overlap":
            client_overlap,
        "test_base_rate":
            base_rate,
        "naive_random_P@20":
            base_rate,
        "precision_at_20":
            p20,
        "precision_at_50":
            p50,
        "lift_at_20": (
            p20 / base_rate
            if base_rate > 0
            else np.nan
        ),
        "average_precision":
            average_precision_score(
                scored_test[LABEL],
                scored_test[
                    "model_score"
                ],
            ),
        "roc_auc":
            roc_auc_score(
                scored_test[LABEL],
                scored_test[
                    "model_score"
                ],
            ),
    })

    split_runs[split_name] = {
        "model": model,
        "train": train_frame,
        "test": test_frame,
        "scored_test": scored_test,
        "client_overlap":
            client_overlap,
    }

split_comparison = pd.DataFrame(
    metric_rows
)

print("BEFORE/AFTER VALIDATION TABLE")

display(
    split_comparison.round(4)
)

assert (
    split_runs[random_name][
        "client_overlap"
    ]
    > 0
)

assert (
    split_runs[grouped_name][
        "client_overlap"
    ]
    == 0
)

assert split_comparison[
    "test_base_rate"
].between(0, 1).all()

assert split_comparison[
    "precision_at_20"
].between(0, 1).all()


# ---------------------------------------------------------
# Result-dependent interpretation
# ---------------------------------------------------------

random_result = (
    split_comparison.loc[
        split_comparison[
            "split_design"
        ]
        == random_name
    ]
    .iloc[0]
)

grouped_result = (
    split_comparison.loc[
        split_comparison[
            "split_design"
        ]
        == grouped_name
    ]
    .iloc[0]
)

display(Markdown(f"""
### Observed before/after result

The random-row diagnostic measured
**Precision@20 = {random_result['precision_at_20']:.3f}**
against a test base rate of
**{random_result['test_base_rate']:.3f}**.
It had **{int(random_result['client_overlap'])} overlapping
clients** between training and test data.

The client-grouped holdout measured
**Precision@20 = {grouped_result['precision_at_20']:.3f}**
against a test base rate of
**{grouped_result['test_base_rate']:.3f}**.
It had **zero client overlap** and measured
**ROC-AUC = {grouped_result['roc_auc']:.3f}**.

The grouped result is the evidence used for this audit
because it measures transfer to unseen pseudonymous clients.
The before/after gap is directional evidence of
client-specific structure and test-population shift.
It does not prove that changing the split changed the
model's true quality.
"""))


# ---------------------------------------------------------
# Real failure examples from grouped holdout
# ---------------------------------------------------------

grouped_scored = rank_for_review(
    split_runs[grouped_name][
        "scored_test"
    ]
)

grouped_scored.insert(
    0,
    "model_rank",
    np.arange(
        1,
        len(grouped_scored) + 1,
    ),
)

grouped_scored[
    "selected_top20"
] = (
    grouped_scored[
        "model_rank"
    ]
    <= 20
)

grouped_scored[
    "error_type"
] = np.select(
    [
        (
            grouped_scored[
                "selected_top20"
            ]
            & (
                grouped_scored[LABEL]
                == 0
            )
        ),
        (
            ~grouped_scored[
                "selected_top20"
            ]
            & (
                grouped_scored[LABEL]
                == 1
            )
        ),
    ],
    [
        "top20_false_positive",
        "proxy_positive_outside_top20",
    ],
    default=
        "correct_for_top20_decision",
)

false_positives = (
    grouped_scored.loc[
        grouped_scored[
            "error_type"
        ]
        == "top20_false_positive"
    ]
    .copy()
)

missed_positives = (
    grouped_scored.loc[
        grouped_scored[
            "error_type"
        ]
        == "proxy_positive_outside_top20"
    ]
    .copy()
)

failure_examples = pd.concat([
    false_positives.head(2),
    missed_positives.head(1),
]).head(3).copy()

assert len(failure_examples) == 3

failure_examples[
    "interpretation"
] = np.where(
    failure_examples[
        "error_type"
    ]
    == "top20_false_positive",

    (
        "Historical signals produced a high "
        "review rank, but no decline was "
        "observed in the short outcome window. "
        "Client-specific demand, seasonality, "
        "or noise may not transfer."
    ),

    (
        "A decline was observed, but the page "
        "ranked below the fixed review capacity "
        "because its historical signals appeared "
        "less risky."
    ),
)

error_summary = pd.DataFrame([{
    "honest_test_rows":
        len(grouped_scored),

    "observed_test_positives":
        int(
            grouped_scored[
                LABEL
            ].sum()
        ),

    "top20_true_positives":
        int(
            grouped_scored
            .head(20)[LABEL]
            .sum()
        ),

    "top20_false_positives":
        int(
            (
                grouped_scored
                .head(20)[LABEL]
                == 0
            ).sum()
        ),

    "proxy_positives_outside_top20":
        len(missed_positives),
}])

print("HONEST-HOLDOUT ERROR SUMMARY")

display(error_summary)

print(
    "THREE REAL PSEUDONYMIZED "
    "FAILURE EXAMPLES"
)

display(
    failure_examples[[
        "model_rank",
        GROUP,
        CONTENT_ID,
        LABEL,
        "model_score",
        "error_type",
        "hist_imp_per_day",
        "hist_click_per_day",
        "hist_ctr",
        "hist_avg_position",
        "hist_position_sd",
        "interpretation",
    ]]
    .round(5)
)

print(
    "PASS: Random-row and grouped-client "
    "splits were compared."
)

print(
    "PASS: Three real pseudonymized "
    "failure examples were inspected."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_rows,pseudonymous_clients,overall_positive_base_rate,feature_count,duplicate_client_content_rows,queried_partition
0,2348,14,0.2355,5,0,2026-03 only


June/_sample queried: NO
BEFORE/AFTER VALIDATION TABLE


,split_design,train_rows,test_rows,train_clients,test_clients,client_overlap,test_base_rate,naive_random_P@20,precision_at_20,precision_at_50,lift_at_20,average_precision,roc_auc
0,Before — random row split (diagnostic only),1761,587,14,12,12,0.2351,0.2351,0.7,0.64,2.9775,0.5390,0.7713
1,After — client-grouped holdout (honest),2120,228,10,4,0,0.0570,0.0570,0.0,0.04,0.0000,0.0461,0.3284



### Observed before/after result

The random-row diagnostic measured
**Precision@20 = 0.700**
against a test base rate of
**0.235**.
It had **12 overlapping
clients** between training and test data.

The client-grouped holdout measured
**Precision@20 = 0.000**
against a test base rate of
**0.057**.
It had **zero client overlap** and measured
**ROC-AUC = 0.328**.

The grouped result is the evidence used for this audit
because it measures transfer to unseen pseudonymous clients.
The before/after gap is directional evidence of
client-specific structure and test-population shift.
It does not prove that changing the split changed the
model's true quality.


HONEST-HOLDOUT ERROR SUMMARY


,honest_test_rows,observed_test_positives,top20_true_positives,top20_false_positives,proxy_positives_outside_top20
0,228,13,0,20,13


THREE REAL PSEUDONYMIZED FAILURE EXAMPLES


,model_rank,client_hash_id,content_hash_id,is_declining_proxy,model_score,error_type,hist_imp_per_day,hist_click_per_day,hist_ctr,hist_avg_position,hist_position_sd,interpretation
0,1,client_fef1a8f436438636,content_0aaa197051f58d6f,0,0.85790,top20_false_positive,2011.31250,1.0625,0.00053,34.73665,5.07147,"Historical signals produced a high review rank, but no decline was observed in the short outcome window. Client-specific demand, seasonality, or noise may not transfer."
1,2,client_fef1a8f436438636,content_7960401fa9cfe53b,0,0.78324,top20_false_positive,977.81250,6.8750,0.00703,37.14976,8.75657,"Historical signals produced a high review rank, but no decline was observed in the short outcome window. Client-specific demand, seasonality, or noise may not transfer."
20,21,client_fef1a8f436438636,content_ff523522181a94f9,1,0.47339,proxy_positive_outside_top20,116.93333,0.8000,0.00684,17.40764,2.58738,"A decline was observed, but the page ranked below the fixed review capacity because its historical signals appeared less risky."


PASS: Random-row and grouped-client splits were compared.
PASS: Three real pseudonymized failure examples were inspected.


## 3. Leakage audit

The Week-5 proxy label is calculated using impressions per day during the outcome window. All five model features are calculated strictly before the decision moment.

I audit three leakage classes:

1. **Label-derived leakage:** The outcome impressions, outcome rate, proxy label, `trend_pct`, `trend_direction`, and any copied label are blocked from the feature list.
2. **Future or overlapping windows:** The feature window ends on 2026-03-21. The label window starts on 2026-03-22, so the two windows do not overlap.
3. **Decision-derived or identity leakage:** No existing product flag, product score, client identifier, or content identifier is used as a model feature.

The pseudonymous client identifier is used only for grouped validation. The content identifier is used only for context and deterministic ranking ties.

The `ga4_data_available IS TRUE` filter reproduces my Week-5 eligible population. It may introduce selection bias regarding which clients and pages are measured, but it does not move future outcome information into the five historical features. I therefore treat this filter as a limitation, not as evidence of causality.

In [7]:
# ---------------------------------------------------------
# Feature provenance audit
# ---------------------------------------------------------

feature_audit = pd.DataFrame([
    {
        "feature":
            "hist_imp_per_day",
        "source_window":
            "2026-03-01 to 2026-03-21",
        "known_at_decision": True,
        "label_or_future_derived": False,
        "decision_or_id_field": False,
        "verdict": "KEEP",
    },
    {
        "feature":
            "hist_click_per_day",
        "source_window":
            "2026-03-01 to 2026-03-21",
        "known_at_decision": True,
        "label_or_future_derived": False,
        "decision_or_id_field": False,
        "verdict": "KEEP",
    },
    {
        "feature":
            "hist_ctr",
        "source_window":
            "2026-03-01 to 2026-03-21",
        "known_at_decision": True,
        "label_or_future_derived": False,
        "decision_or_id_field": False,
        "verdict": "KEEP",
    },
    {
        "feature":
            "hist_avg_position",
        "source_window":
            "2026-03-01 to 2026-03-21",
        "known_at_decision": True,
        "label_or_future_derived": False,
        "decision_or_id_field": False,
        "verdict": "KEEP",
    },
    {
        "feature":
            "hist_position_sd",
        "source_window":
            "2026-03-01 to 2026-03-21",
        "known_at_decision": True,
        "label_or_future_derived": False,
        "decision_or_id_field": False,
        "verdict": "KEEP",
    },
])

blocked_fields = {
    "outcome_days",
    "outcome_impressions",
    "outcome_imp_per_day",
    "is_declining_proxy",
    "trend_pct",
    "trend_direction",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "LEAK_label_copy",
}

blocked_intersection = sorted(
    set(FEATURES).intersection(
        blocked_fields
    )
)

timeline = pd.DataFrame([
    {
        "role": "model features",
        "start":
            pd.Timestamp("2026-03-01"),
        "end":
            pd.Timestamp("2026-03-21"),
        "available_at_decision": True,
    },
    {
        "role": "outcome label",
        "start":
            pd.Timestamp("2026-03-22"),
        "end":
            pd.Timestamp("2026-03-31"),
        "available_at_decision": False,
    },
])

feature_end = (
    timeline.loc[
        timeline["role"]
        == "model features",
        "end",
    ]
    .iloc[0]
)

label_start = (
    timeline.loc[
        timeline["role"]
        == "outcome label",
        "start",
    ]
    .iloc[0]
)

print("FEATURE PROVENANCE AUDIT")

display(feature_audit)

print("WINDOW ALIGNMENT")

display(timeline)

print(
    "Blocked fields found in "
    "feature list:",
    blocked_intersection,
)

assert (
    feature_audit[
        "feature"
    ].tolist()
    == FEATURES
)

assert feature_audit[
    "known_at_decision"
].all()

assert not feature_audit[
    "label_or_future_derived"
].any()

assert not feature_audit[
    "decision_or_id_field"
].any()

assert set(
    feature_audit["verdict"]
) == {"KEEP"}

assert feature_end < label_start
assert blocked_intersection == []
assert LABEL not in FEATURES
assert GROUP not in FEATURES
assert CONTENT_ID not in FEATURES


# ---------------------------------------------------------
# Deliberate leakage harness
# ---------------------------------------------------------
# This experiment is intentionally invalid.
# It verifies that the evaluation harness reacts when
# the answer is copied into the input. Its result must
# never be reported as model performance.

honest_train = (
    split_runs[grouped_name][
        "train"
    ]
    .copy()
)

honest_test = (
    split_runs[grouped_name][
        "test"
    ]
    .copy()
)

leak_feature = "LEAK_label_copy"

honest_train[
    leak_feature
] = honest_train[LABEL]

honest_test[
    leak_feature
] = honest_test[LABEL]

leaky_features = (
    FEATURES
    + [leak_feature]
)

leaky_model = make_logistic_model()

leaky_model.fit(
    honest_train[leaky_features],
    honest_train[LABEL],
)

leaky_scores = (
    leaky_model.predict_proba(
        honest_test[leaky_features]
    )[:, 1]
)

honest_grouped_row = (
    split_comparison.loc[
        split_comparison[
            "split_design"
        ]
        == grouped_name
    ]
    .iloc[0]
)

leakage_harness = pd.DataFrame([
    {
        "run":
            "Honest grouped model",
        "feature_set":
            "Five historical features",
        "valid_evidence": True,
        "average_precision":
            honest_grouped_row[
                "average_precision"
            ],
        "roc_auc":
            honest_grouped_row[
                "roc_auc"
            ],
    },
    {
        "run":
            "Intentionally invalid leak test",
        "feature_set":
            "Historical features + copied label",
        "valid_evidence": False,
        "average_precision":
            average_precision_score(
                honest_test[LABEL],
                leaky_scores,
            ),
        "roc_auc":
            roc_auc_score(
                honest_test[LABEL],
                leaky_scores,
            ),
    },
])

print(
    "LEAKAGE HARNESS — SECOND ROW "
    "IS INTENTIONALLY INVALID"
)

display(
    leakage_harness.round(4)
)

invalid_row = (
    leakage_harness.loc[
        ~leakage_harness[
            "valid_evidence"
        ]
    ]
    .iloc[0]
)

assert invalid_row["roc_auc"] > 0.999

assert (
    invalid_row[
        "average_precision"
    ]
    > 0.999
)

assert leak_feature not in FEATURES

print(
    "PASS: Label-derived, future-window, "
    "product-decision, and ID leakage "
    "checks passed."
)

print(
    "PASS: The deliberately copied label "
    "produced a near-perfect invalid score."
)

print(
    "PASS: The copied label remains "
    "excluded from the honest features."
)

FEATURE PROVENANCE AUDIT


,feature,source_window,known_at_decision,label_or_future_derived,decision_or_id_field,verdict
0,hist_imp_per_day,2026-03-01 to 2026-03-21,True,False,False,KEEP
1,hist_click_per_day,2026-03-01 to 2026-03-21,True,False,False,KEEP
2,hist_ctr,2026-03-01 to 2026-03-21,True,False,False,KEEP
3,hist_avg_position,2026-03-01 to 2026-03-21,True,False,False,KEEP
4,hist_position_sd,2026-03-01 to 2026-03-21,True,False,False,KEEP


WINDOW ALIGNMENT


,role,start,end,available_at_decision
0,model features,2026-03-01,2026-03-21,True
1,outcome label,2026-03-22,2026-03-31,False


Blocked fields found in feature list: []
LEAKAGE HARNESS — SECOND ROW IS INTENTIONALLY INVALID


,run,feature_set,valid_evidence,average_precision,roc_auc
0,Honest grouped model,Five historical features,True,0.0461,0.3284
1,Intentionally invalid leak test,Historical features + copied label,False,1.0000,1.0000


PASS: Label-derived, future-window, product-decision, and ID leakage checks passed.
PASS: The deliberately copied label produced a near-perfect invalid score.
PASS: The copied label remains excluded from the honest features.


## 4. Claim rewrite

### Original Week-5 sentence

> “The simpler Logistic Regression remains preferred.”

This sentence follows my Week-5 model-selection rule, but the word `preferred` can sound broader than the evidence.

The grouped holdout covers one March 2026 slice, a short proxy-outcome window, and four unseen pseudonymous clients. Its test population also has a different positive base rate from the full eligible frame.

The model was evaluated using observed page-level outcomes. It was not tested as an intervention, so it cannot establish that refreshing a selected page would cause traffic recovery.

The code below rewrites the claim using the measured grouped result and the required public-safe terms: **observed**, **measured**, **directional**, and **decision-support**.

In [8]:
grouped_row = (
    split_comparison.loc[
        split_comparison[
            "split_design"
        ]
        == grouped_name
    ]
    .iloc[0]
)

safe_claim = (
    f"In one observed March 2026 "
    f"client-grouped holdout containing "
    f"{int(grouped_row['test_rows'])} rows "
    f"from {int(grouped_row['test_clients'])} "
    f"unseen pseudonymous clients, Logistic "
    f"Regression measured Precision@20 = "
    f"{grouped_row['precision_at_20']:.3f} "
    f"against a test base rate of "
    f"{grouped_row['test_base_rate']:.3f}, "
    f"with ROC-AUC = "
    f"{grouped_row['roc_auc']:.3f}. "
    f"This is directional evidence from one "
    f"short proxy window, not evidence that "
    f"a refresh causes traffic recovery. "
    f"The model remains a human-reviewed "
    f"decision-support experiment and is not "
    f"validated for automatic decisions or "
    f"deployment."
)

claim_audit = pd.DataFrame([
    {
        "version": "Original",
        "claim":
            "The simpler Logistic "
            "Regression remains preferred.",
        "status":
            "Too broad without the "
            "measured holdout context",
    },
    {
        "version":
            "Public-safe rewrite",
        "claim":
            safe_claim,
        "status":
            "Bounded to the observed "
            "design and decision-support use",
    },
])

display(claim_audit)

required_words = [
    "observed",
    "measured",
    "directional",
    "decision-support",
]

for word in required_words:
    assert word in safe_claim.lower()

assert "not evidence" in safe_claim.lower()
assert "not validated" in safe_claim.lower()


# ---------------------------------------------------------
# Machine-verifiable final checks
# ---------------------------------------------------------

self_checks = pd.DataFrame([
    {
        "check":
            "Exactly two paper findings audited",
        "passed":
            len(paper_audit) == 2,
    },
    {
        "check":
            "Week-5 row count reproduced",
        "passed":
            len(model_frame) == 2348,
    },
    {
        "check":
            "Random split has client overlap",
        "passed":
            split_runs[random_name][
                "client_overlap"
            ] > 0,
    },
    {
        "check":
            "Grouped split has zero overlap",
        "passed":
            split_runs[grouped_name][
                "client_overlap"
            ] == 0,
    },
    {
        "check":
            "Base rate reported for both splits",
        "passed":
            split_comparison[
                "test_base_rate"
            ].notna().all(),
    },
    {
        "check":
            "Exactly five historical features",
        "passed":
            len(FEATURES) == 5,
    },
    {
        "check":
            "No blocked feature used",
        "passed":
            blocked_intersection == [],
    },
    {
        "check":
            "Feature and label windows do not overlap",
        "passed":
            feature_end < label_start,
    },
    {
        "check":
            "Copied-label leakage detected",
        "passed":
            invalid_row["roc_auc"] > 0.999,
    },
    {
        "check":
            "Three failure examples inspected",
        "passed":
            len(failure_examples) == 3,
    },
    {
        "check":
            "Safe claim contains required language",
        "passed":
            all(
                word in safe_claim.lower()
                for word in required_words
            ),
    },
])

display(self_checks)

assert self_checks[
    "passed"
].all()

reproducibility = pd.DataFrame([{
    "random_seed": SEED,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn":
        sklearn.__version__,
    "duckdb": duckdb.__version__,
}])

display(reproducibility)

print(
    "ALL MACHINE-VERIFIABLE "
    "CHECKS PASSED."
)

,version,claim,status
0,Original,The simpler Logistic Regression remains preferred.,Too broad without the measured holdout context
1,Public-safe rewrite,"In one observed March 2026 client-grouped holdout containing 228 rows from 4 unseen pseudonymous clients, Logistic Regression measured Precision@20 = 0.000 against a test base ...",Bounded to the observed design and decision-support use


,check,passed
0,Exactly two paper findings audited,True
1,Week-5 row count reproduced,True
2,Random split has client overlap,True
3,Grouped split has zero overlap,True
4,Base rate reported for both splits,True
5,Exactly five historical features,True
6,No blocked feature used,True
7,Feature and label windows do not overlap,True
8,Copied-label leakage detected,True
9,Three failure examples inspected,True


,random_seed,numpy,pandas,scikit_learn,duckdb
0,42,2.0.2,2.2.2,1.6.1,1.3.2


ALL MACHINE-VERIFIABLE CHECKS PASSED.


## 5. Self-check

- [x] Every section contains markdown reasoning and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] Two paper findings are named and reviewed constructively.
- [x] The source of each paper label or exposure is explained.
- [x] The random-row and client-grouped results are compared.
- [x] The base rate appears beside the evaluation metrics.
- [x] The grouped split has zero client overlap.
- [x] The Week-5 frame reproduces 2,348 eligible rows and 14 pseudonymous clients.
- [x] Exactly five historical features are used.
- [x] Three real pseudonymized failure examples are displayed and interpreted.
- [x] The feature and outcome windows do not overlap.
- [x] Outcome fields, labels, product flags, and identifiers are excluded from the honest features.
- [x] The deliberately invalid copied-label test detects leakage.
- [x] The copied label is removed from the honest feature list.
- [x] The final claim uses observed, measured, directional, and decision-support language.
- [x] No causal recovery or automatic-deployment claim is made.
- [x] June 2026 and the `_sample` answer-key table were not queried.
- [x] No token, client name, URL, title, or private query is stored or printed.
- [x] Saved and committed as `work/notebooks/w06_validation_audit.ipynb`.